# NSO / IEEE BigDataCup 2026: complete Solar filament inference reproduction

**Purpose:** reproduce the competition submission from **original organizer test JPEGs** and a genuine trained compact U-Net checkpoint. No hidden test labels are used. The network predicts 512px probabilities with four-flip TTA; each instance is upscaled to 2048px, edge-clipped and COCO RLE encoded. Full-test CSV SHA-256 must equal the published candidate, otherwise execution **fails**.

The organizer also requires a separate four-page report, a publicly available Git repository and a final submission Google Form. Running this notebook is **not** an official Kaggle submission or score. Data owners should use their already-authorized competition files. Kaggle original `/kaggle/input/` folders, or the supplied `MAGFiLO_1.0_Kaggle_2026.zip`, both work. The included checkpoint means **no extra model file** is required.

In [1]:
from pathlib import Path
import os,sys,json,hashlib,importlib.metadata as md
ROOT = Path.cwd()
if not (ROOT/'solar_real.py').exists():
    ROOT = ROOT.parent  # also works if Jupyter was launched from notebooks/
assert (ROOT/'solar_real.py').exists(), 'Launch from the repository root or notebooks/ directory'
sys.path.insert(0,str(ROOT))
print('project',ROOT)
for package in ('numpy','scipy','torch','opencv-python','Pillow'):
    print(f'{package}: {md.version(package)}')
assert (ROOT/'results/SOLAR_REAL_TRAINED_UNET.pt').is_file()


project /mnt/data/Solar_Filament_2026_FINAL_READY_20261105
numpy: 2.3.5
scipy: 1.17.0
torch: 2.10.0+cpu
opencv-python: 4.13.0.92
Pillow: 12.3.0


## 1. Access the real, unlabeled test JPEGs

Test files are supplied through the competition. Set `SOLAR_ZIP2` to the participant's original Kaggle ZIP, **or** set `SOLAR_TEST_DIR` to the extracted MAGFiLO dataset directory. On Kaggle, attaching the original competition dataset under `/kaggle/input` also works. The following cell raises an error rather than silently skipping the missing data.

In [2]:
from solar_source import OfficialTestImages
candidates = [os.environ.get('SOLAR_ZIP2'),os.environ.get('SOLAR_TEST_DIR')]
if Path('/kaggle/input').is_dir():candidates.append('/kaggle/input')
candidates.extend([str(ROOT/'MAGFiLO_1.0_Kaggle_2026.zip'),str(ROOT/'MAGFiLO_1.0_Kaggle_2026')])
test_source = None
for loc in candidates:
    if not loc:continue
    try:
        with OfficialTestImages(loc) as read:
            ids=list(read.paths)
            assert len(ids)==180
            test_source=Path(loc)
        break
    except (ValueError,FileNotFoundError,NotADirectoryError):
        pass
assert test_source is not None, 'Missing original 180 official test JPEGs; attach the competition dataset or set SOLAR_ZIP2/SOLAR_TEST_DIR'
print('verified 180 official test images from',test_source.name)
print('example official IDs:',ids[:3])


verified 180 official test images from _kaggle_input_data
example official IDs: ['20110120105534Ch', '20110130110334Ch', '20110214175414Mh']


## 2. Read the actual saved model and visualize its real test-image prediction

This is the original model fitted on 707 authorized MAGFiLO train photos (582 fitting, 63 tuning, 62 date-separated local audit); the three-dimensional segmentation target is produced by connected components, not by a separate instance-classifier. All model decisions use the saved checkpoint and tune-only threshold configuration.

In [3]:
import cv2,numpy as np,torch,matplotlib.pyplot as plt
from solar_real import read_model,RESULT
from solar_tta import predict_tta
from solar_eval import instance_map
from refined_inference import edge_refine
model=read_model()
configuration=json.loads((RESULT/'SOLAR_TTA_COMPARISON.json').read_text())['parameters']
print('loaded model',model.__class__.__name__,'model SHA256:',hashlib.sha256((RESULT/'SOLAR_REAL_TRAINED_UNET.pt').read_bytes()).hexdigest())
with OfficialTestImages(test_source) as images:
    sample_id=next(iter(images.paths));jpeg=images.bytes(sample_id)
photo=cv2.imdecode(np.frombuffer(jpeg,np.uint8),cv2.IMREAD_GRAYSCALE)
assert photo.shape==(2048,2048)
small=cv2.resize(photo,(512,512),interpolation=cv2.INTER_AREA)
probs=predict_tta(model,small)
instance512=instance_map(probs,small,**configuration)
edge2048=edge_refine(instance512,probs,threshold=.45)
fig,axes=plt.subplots(1,3,figsize=(12,4))
for a,title,picture in zip(axes,['Original GONG H-alpha (512 preview)','4-flip probability','Edge-refined instances (512 preview)'],
                           [small,probs,cv2.resize(edge2048.astype(np.float32),(512,512),interpolation=cv2.INTER_NEAREST)]):
    a.imshow(picture,cmap='gray' if a is axes[0] else 'magma')
    a.set_title(title,fontsize=9);a.axis('off')
fig.tight_layout();plt.show()
print('example',sample_id,'predicted instances',int(instance512.max()))


loaded model FilamentUNet model SHA256: a0183861865e2cb3e152b8640bcc61c503d66b206c9fcfc9ed682a6c7fe6349f


[Organizer-supplied test JPEG preview omitted from public source archive]

example 20110120105534Ch predicted instances 2


## 3. Reproduce all 180 test images, from the trained checkpoint

Each **missing** test image is actually inferred and committed as an atomically written, CRC+SHA-256 checked shard. Previously computed, independently verified shards are reused without expensive retraining. The final complete CSV is committed only if all 180 images pass verification. SHA comparison to the supplied original candidate is mandatory. A fresh clone with no shards will compute all 180 images by itself.

In [4]:
from reproduce_submission import reproduce
from verify_candidate import verify
candidate=RESULT/'SOLAR_MAGFiLO_2026_TTA_NATIVE_EDGE_180_CANDIDATE.csv'
assert candidate.exists(), 'The exact competition candidate should be tracked with this repository'
regenerated=RESULT/'SOLAR_REPRODUCED_FROM_OFFICIAL_JPEGS.csv'
result=reproduce(source_zip=test_source,output=regenerated,
                 shard_dir=ROOT/'_repro_shards',reference=candidate)
assert result['selected_images']==180 and result['same_as_reference'] is True
assert result['csv_rows']==1013 and result['images_without_predictions']==6
print('Full test output has been regenerated, independently decoded and checked:',result['candidate_sha256'])


REPRODUCE 30/180; freshly_inferred=0, verified_reused=30


REPRODUCE 60/180; freshly_inferred=0, verified_reused=60


REPRODUCE 90/180; freshly_inferred=0, verified_reused=90


REPRODUCE 120/180; freshly_inferred=0, verified_reused=120


REPRODUCE 150/180; freshly_inferred=0, verified_reused=150


REPRODUCE 180/180; freshly_inferred=0, verified_reused=180


REPRODUCTION COMPLETE: {
  "selected_images": 180,
  "newly_inferred": 0,
  "existing_sha_verified_shards": 180,
  "official_test_JPEG_zip": "_kaggle_input_data",
  "trained_checkpoint_sha256": "a0183861865e2cb3e152b8640bcc61c503d66b206c9fcfc9ed682a6c7fe6349f",
  "test_ground_truth_used": false,
  "reference_csv": "SOLAR_MAGFiLO_2026_TTA_NATIVE_EDGE_180_CANDIDATE.csv",
  "output_csv": "/mnt/data/Solar_Filament_2026_FINAL_READY_20261105/results/SOLAR_REPRODUCED_FROM_OFFICIAL_JPEGS.csv",
  "same_as_reference": true,
  "csv_rows": 1013,
  "candidate_sha256": "e0e55cf4eb383769c17b742d556d619859307b82a4267792ad2345698126d9ea",
  "images_without_predictions": 6,
  "real_2048_masks_decoded": 30,
  "atomic_commit": true
}


Full test output has been regenerated, independently decoded and checked: e0e55cf4eb383769c17b742d556d619859307b82a4267792ad2345698126d9ea


## 4. Submission integrity and official-format checks

One row per predicted filament; `filament_id` uses the unmodified organizer image ID plus a unique suffix. `segmentation_rle` contains compressed COCO-style counts only (not dimensions). No annotation information or external metadata is available at test-time.

In [5]:
checked=verify(regenerated,test_source,decode_count=30)
assert checked['images']==180 and checked['rows']==1013
assert checked['sha256']==result['candidate_sha256']
assert checked['all_masks_have_correct_shape'] is True
print(json.dumps(checked,indent=2))


{
  "images": 180,
  "rows": 1013,
  "empty_images": 6,
  "sha256": "e0e55cf4eb383769c17b742d556d619859307b82a4267792ad2345698126d9ea",
  "all_masks_have_correct_shape": true,
  "decoded_masks": 30,
  "official_score": null,
  "submitted": false
}


## 5. Model training methodology and measured local validation

A compact eight-channel U-Net was fitted for ten epochs with positive-centered 256x256 crops, brightness/flip augmentation, AdamW, weighted BCE and soft Dice. All 1,154 annotation views remain date-disjoint. The following evidence report covers **original 2048x2048 training/validation polygons**, not the hidden Kaggle test labels. To **refit** instead of using the included checkpoint, set `SOLAR_ZIP1`, `SOLAR_ZIP2` and run `solar_real.cache(); solar_real.train(epochs=10)`. Refitting is optional because it changes the checkpoint and is **not needed to reproduce the submitted CSV**.

In [6]:
evidence=json.loads((RESULT/'SOLAR_ORIGINAL2048_EDGE_REPRODUCED.json').read_text())
assert evidence['n_images']==62 and evidence['n_rater_views']==93
assert evidence['independent_recomputed_TTA_matches_v5']
pq0=evidence['baseline_TTA']['pq'];pq1=evidence['edge_refined_TTA']['pq']
print('Local full-resolution PQ: TTA',round(pq0,6),'edge-refined',round(pq1,6),
      'improvement',round(pq1-pq0,6),'local bootstrap CI',evidence['bootstrap_95_percentile_CI'])
print('This is previously inspected date-separated training audit, NOT an official Kaggle score.')


Local full-resolution PQ: TTA 0.262893 edge-refined 0.273115 improvement 0.010221 local bootstrap CI [0.0016871766187005713, 0.01867409170976381]
This is previously inspected date-separated training audit, NOT an official Kaggle score.


## 6. Reproduction proof and final human submission tasks

This notebook has no Kaggle account access; it cannot register a team, publish your repository, or submit the final Google Form. The participant must complete those account-specific actions using the exact artifact from this notebook. The project includes `results/SOLAR_TTA_REAL_4PAGE_TECHNICAL_DRAFT.pdf`; the participant should review identity/attribution and format against the organizers' Overleaf template before final form submission.

In [7]:
proof={
  'verified_real_test_images':180,'actual_inference_pipeline':'genuine compact U-Net + 4-flip TTA + 2048 edge refinement',
  'model_sha256':hashlib.sha256((RESULT/'SOLAR_REAL_TRAINED_UNET.pt').read_bytes()).hexdigest(),
  'csv_sha256':result['candidate_sha256'],'equals_published_candidate':result['same_as_reference'],
  'individual_filament_rows':result['csv_rows'],'test_ground_truth_used':False,
  'local_training_audit_pq':pq1,'organizer_leaderboard_score':None,
  'formal_submission_completed':False
}
output=RESULT/'SOLAR_NOTEBOOK_EXECUTION_RECEIPT.json'
output.write_text(json.dumps(proof,indent=2)+'\n')
print(json.dumps(proof,indent=2))


{
  "verified_real_test_images": 180,
  "actual_inference_pipeline": "genuine compact U-Net + 4-flip TTA + 2048 edge refinement",
  "model_sha256": "a0183861865e2cb3e152b8640bcc61c503d66b206c9fcfc9ed682a6c7fe6349f",
  "csv_sha256": "e0e55cf4eb383769c17b742d556d619859307b82a4267792ad2345698126d9ea",
  "equals_published_candidate": true,
  "individual_filament_rows": 1013,
  "test_ground_truth_used": false,
  "local_training_audit_pq": 0.27311477742304735,
  "organizer_leaderboard_score": null,
  "formal_submission_completed": false
}
